In [ ]:
!pip install beautifulsoup4

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
!pip install konlpy

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 47.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.3/465.3 kB 29.6 MB/s eta 0:00:00


In [ ]:
!pip install fake_useragent

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.1 MB/s eta 0:00:00


In [ ]:
!pip install finance-datareader

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import pandas as pd
import requests
import re
from bs4 import BeautifulSoup as bs
from fake_useragent import UserAgent
import plotly.express as px
from collections import Counter
from konlpy.tag import Hannanum
from datetime import datetime
import time
import FinanceDataReader as fdr
from tqdm import tqdm

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

hannanum = Hannanum() 

In [ ]:
def crawler(maxpage, query, s_date, e_date):
    """
    검색어를 바탕으로 특정 기간에 해당하는 네이버 뉴스의 날짜, 제목, 본문, 링크를 가져옴
    """
    naver_urls = []
    s_from = s_date.replace(".", "")
    e_to = e_date.replace(".", "")
    page = 1
    maxpage_t =(int(maxpage)-1)*10+1    
    
    while page < maxpage_t:
        
        url = "https://search.naver.com/search.naver?where=news&sm=tab_pge&query=" + query + "&ds=" + s_date + "&de=" + e_date +  "&nso=so%3Ar%2Cp%3Afrom" + s_from + "to" + e_to + "%2Ca%3A&start=" + str(page)
        # ua = UserAgent()
        # headers = {'User-Agent' : ua.random}

        req = requests.get(url)
        
        cont = req.content
        soup = BeautifulSoup(cont, 'html.parser')
        
        for urls in soup.select("a.info"):


            try:
                if "news.naver.com" in urls['href']:
                    naver_urls.append(urls['href'])
                  
            except Exception as e:
                continue
        page += 10
        
    return naver_urls
    
max_page = 20              ## 크롤링 할 최대 페이지
query = '집값'           ## 섹터별 단어 설정
s_date = '2021.01.01'
e_date = '2021.02.01'
year = ['2021']
month = ['01','02']
for j in list(range(0,1)):
    s_date = s_date.split(".")
    s_date[0] = year[j]
    s_date = ".".join(s_date)
    
    e_date = e_date.split(".")
    e_date[0] = year[j]
    e_date = ".".join(e_date)
    for i in list(range(0,2)):
        s_date = s_date.split(".")
        s_date[1] = month[i]
        s_date =  ".".join(s_date)

        
        e_date = e_date.split(".")
        e_date[1] = month[i]
        e_date =  ".".join(e_date)
        
        total_urls = crawler(max_page,query,s_date,e_date)
        
        
        total_urls = list(set(total_urls))
        
        # ConnectionError방지

        headers = { "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/98.0.4758.102" }


        titles = []
        contents=[]
        dates = []
        result_url = []
        for i in tqdm(total_urls):
            original_html = requests.get(i,headers=headers)
            html = BeautifulSoup(original_html.text, "html.parser")
            
            ## 날짜가져오기
            try :
                html_date = html.select_one("div#ct> div.media_end_head.go_trans > div.media_end_head_info.nv_notrans > div.media_end_head_info_datestamp > div > span")
                news_date = html_date.attrs['data-date-time']
                dates.append(news_date)
            except :
                continue
            result_url.append(i)


            #뉴스 제목 가져오기
            title = html.select("div#ct > div.media_end_head.go_trans > div.media_end_head_title > h2")
            # list합치기
            title = ''.join(str(title))
            # html태그제거
            pattern1 = '<[^>]*>'
            title = re.sub(pattern=pattern1,repl='',string=title)
            titles.append(title)

            #뉴스 본문 가져오기

            content = html.select("div#dic_area")

            # 기사 텍스트만 가져오기
            # list합치기
            content = ''.join(str(content))

            #html태그제거 및 텍스트 다듬기
            content = re.sub(pattern=pattern1,repl='',string=content)
            pattern2 = """[\n\n\n\n\n// flash 오류를 우회하기 위한 함수 추가\nfunction _flash_removeCallback() {}"""
            content = content.replace(pattern2,'')

            contents.append(content)



        news_df = pd.DataFrame({'date' : dates,'title': titles, 'link': result_url, 'content': contents})
        news_df.to_csv(f"news_{query}from_{s_date}to_{e_date}.csv" ,index=False, encoding='utf-8-sig')

NameError: ignored

In [ ]:
news_df